# Download data from a STAC API using R, rstac, and GDAL

This tutorial walks through querying a STAC API using the
[rstac](https://brazil-data-cube.github.io/rstac/) R package, and
downloading data from the API using rstac or [GDAL](https://gdal.org/)
(via [sf](https://github.com/r-spatial/sf)). This tutorial will assume
that you’re already familiar with [R](https://cran.r-project.org/) and
working with spatial data.

As all of the packages we’ll be using are available from CRAN, you can
install them (if necessary) using `install.packages()`:

In [1]:
install.packages("units", verbose = TRUE)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

system (cmd0): /usr/local/lib/R/bin/R CMD INSTALL

also installing the dependency ‘Rcpp’


foundpkgs: Rcpp, units, /tmp/Rtmp0VtriZ/downloaded_packages/Rcpp_1.0.14.tar.gz, /tmp/Rtmp0VtriZ/downloaded_packages/units_0.8-7.tar.gz

files: /tmp/Rtmp0VtriZ/downloaded_packages/Rcpp_1.0.14.tar.gz, 
	/tmp/Rtmp0VtriZ/downloaded_packages/units_0.8-7.tar.gz

1): succeeded '/usr/local/lib/R/bin/R CMD INSTALL -l '/usr/local/lib/R/site-library' '/tmp/Rtmp0VtriZ/downloaded_packages/Rcpp_1.0.14.tar.gz''

2): succeeded '/usr/local/lib/R/bin/R CMD INSTALL -l '/usr/local/lib/R/site-library' '/tmp/Rtmp0VtriZ/downloaded_packages/units_0.8-7.tar.gz''



In [2]:
install.packages("s2", verbose = TRUE)
# install.packages("s2", type = "source", verbose = TRUE)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

system (cmd0): /usr/local/lib/R/bin/R CMD INSTALL

also installing the dependency ‘wk’


foundpkgs: wk, s2, /tmp/Rtmp0VtriZ/downloaded_packages/wk_0.9.4.tar.gz, /tmp/Rtmp0VtriZ/downloaded_packages/s2_1.1.7.tar.gz

files: /tmp/Rtmp0VtriZ/downloaded_packages/wk_0.9.4.tar.gz, 
	/tmp/Rtmp0VtriZ/downloaded_packages/s2_1.1.7.tar.gz

1): succeeded '/usr/local/lib/R/bin/R CMD INSTALL -l '/usr/local/lib/R/site-library' '/tmp/Rtmp0VtriZ/downloaded_packages/wk_0.9.4.tar.gz''

2): succeeded '/usr/local/lib/R/bin/R CMD INSTALL -l '/usr/local/lib/R/site-library' '/tmp/Rtmp0VtriZ/downloaded_packages/s2_1.1.7.tar.gz''



In [1]:
# install.packages("sf")
install.packages("sf", verbose = TRUE)

# configure: error: gdal-config not found or not executable.

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

system (cmd0): /usr/local/lib/R/bin/R CMD INSTALL

also installing the dependencies ‘proxy’, ‘e1071’, ‘wk’, ‘classInt’, ‘DBI’, ‘magrittr’, ‘s2’, ‘units’, ‘Rcpp’


foundpkgs: proxy, e1071, wk, classInt, DBI, magrittr, s2, units, Rcpp, sf, /tmp/RtmpgQ2u8Z/downloaded_packages/proxy_0.4-27.tar.gz, /tmp/RtmpgQ2u8Z/downloaded_packages/e1071_1.7-16.tar.gz, /tmp/RtmpgQ2u8Z/downloaded_packages/wk_0.9.4.tar.gz, /tmp/RtmpgQ2u8Z/downloaded_packages/classInt_0.4-11.tar.gz, /tmp/RtmpgQ2u8Z/downloaded_packages/DBI_1.2.3.tar.gz, /tmp/RtmpgQ2u8Z/downloaded_packages/magrittr_2.0.3.tar.gz, /tmp/RtmpgQ2u8Z/downloaded_packages/s2_1.1.7.tar.gz, /tmp/RtmpgQ2u8Z/downloaded_packages/units_0.8-7.tar.gz, /tmp/RtmpgQ2u8Z/downloaded_packages/Rcpp_1.0.14.tar.gz, /tmp/RtmpgQ2u8Z/downloaded_packages/sf_1.0-19.tar.gz

files: /tmp/RtmpgQ2u8Z/downloaded_packages/proxy_0.4-27.tar.gz, 
	/tmp/RtmpgQ2u8Z/downloaded_packages/e1071_1.7-16.tar.g

In [2]:
install.packages("rstac")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘sys’, ‘askpass’, ‘curl’, ‘mime’, ‘openssl’, ‘R6’, ‘httr’, ‘png’, ‘jpeg’




In [3]:
install.packages("terra")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



STAC APIs are servers that provide access to a set of data which users can query and retrieve. You can find a partial list of STAC APIs at [STAC Index, https://stacindex.org/](https://stacindex.org/). For this tutorial, we’ll be downloading data from Microsoft’s [Planetary Computer](https://planetarycomputer.microsoft.com), which currently provides more than 100 data sets for free via a central STAC API.

To start downloading data, we’ll first need to let rstac know what STAC API we want to query and download from. To do so, we’ll pass the URL of the Planetary Computer STAC API to the `rstac::stac()` function:

In [4]:
stac_source <- rstac::stac(
  "https://planetarycomputer.microsoft.com/api/stac/v1"
)
stac_source

###rstac_query
- url: https://planetarycomputer.microsoft.com/api/stac/v1/
- params:
- field(s): version, base_url, endpoint, params, verb, encode

As you can see, the output of `stac()` is an `RSTACQuery` object, which
contains information about an HTTP query that we might want to run in
the future. Under the hood, these objects are normal lists containing
information about the query:

In [5]:
str(stac_source)

List of 6
 $ version : NULL
 $ base_url: chr "https://planetarycomputer.microsoft.com/api/stac/v1/"
 $ endpoint: NULL
 $ params  : list()
 $ verb    : chr "GET"
 $ encode  : NULL
 - attr(*, "class")= chr [1:2] "stac" "rstac_query"


But most of the time, you won’t need to worry about this internal
representation; rstac provides many helper functions to access the
elements of this list if needed.

It’s worth highlighting that this object is a representation of a
*future* HTTP query, not the results of a query we’ve already run! In
order to actually run these queries, we need to use
`rstac::get_request()` (or `rstac::post_request()`, depending on what
HTTP verb your STAC API is expecting). If we use `get_request()` to
query the Planetary Computer STAC API, we get a brief description of
what this API provides:

In [6]:
rstac::get_request(stac_source)

###Catalog
- id: microsoft-pc
- description: 
Searchable spatiotemporal metadata describing Earth science datasets hosted by the Microsoft Planetary Computer
- field(s): 
type, id, title, description, stac_version, conformsTo, links, stac_extensions

Because the `RSTACQuery` object is a representation of a future query,
we can use other functions in rstac to change our query parameters and
fields before we actually make a request. For instance, we can use
`rstac::collections()` to update our request to query the `/collections`
endpoint of the Planetary Computer API, which lists all the available
[collections](https://github.com/radiantearth/stac-spec/blob/master/collection-spec/collection-spec.md)
(which, to quote the STAC Collection specification, “describe a group of
[Items](https://github.com/radiantearth/stac-spec/blob/master/item-spec/item-spec.md)
that share properties and metadata”):

In [7]:
collections_query <- stac_source |>
  rstac::collections()

collections_query

###rstac_query
- url: https://planetarycomputer.microsoft.com/api/stac/v1/
- params:
- field(s): version, base_url, endpoint, params, verb, encode

While it might not look like much has changed, under the hood our
`collections_query` object has a new `collections` class to indicate
that we’re querying the collections endpoint, not the top-level STAC
endpoint:

In [8]:
class(stac_source)

[1] "stac"        "rstac_query"

In [9]:
class(collections_query)

[1] "collections" "rstac_query"

And as a result, when we use `get_request()` to turn this query
*specification* into a query *result*, we get a list of the collections
available from this API:

In [10]:
available_collections <- rstac::get_request(collections_query)
available_collections

###Collections
- collections (126 item(s)):
  - daymet-annual-pr
  - daymet-daily-hi
  - 3dep-seamless
  - 3dep-lidar-dsm
  - fia
  - sentinel-1-rtc
  - gridmet
  - daymet-annual-na
  - daymet-monthly-na
  - daymet-annual-hi
  - ... with 116 more collection(s).
- field(s): collections, links

These collections are a subset of the data sets available in the
[Planetary Computer data
catalog](https://planetarycomputer.microsoft.com/catalog); the Planetary
Computer is organized so that each collection corresponds to a distinct
data set in the catalog.

For our purposes today, we’re going to be querying the [USGS Land Change
Monitoring, Assessment, and Projection
(LCMAP)](https://planetarycomputer.microsoft.com/dataset/usgs-lcmap-conus-v13)
collection, which provides (among other things) annual land cover
classifications for the continental United States. We can query what
items are available for this collection using the `rstac::stac_search()`
function. We'll limit our search to 2021 using the `datetime` argument, 
ask for up to 999 items (the most Planetary Computer will return in a 
single request) using the `limit` argument, and only search within the 
LCMAP collection by using the collection’s ID of `usgs-lcmap-conus-v13`:

In [11]:
rstac::stac_search(
  q = stac_source,
  collections = "usgs-lcmap-conus-v13",
  datetime = "2021-01-01/2021-12-31",
  limit = 999
)

###rstac_query
- url: https://planetarycomputer.microsoft.com/api/stac/v1/
- params:
  - collections: usgs-lcmap-conus-v13
  - datetime: 2021-01-01/2021-12-31
  - limit: 999
- field(s): version, base_url, endpoint, params, verb, encode

We can see that there are 422 items inside this catalog for 2021.
Collectively, these items contain all LCMAP data for the continental
United States for 2021, with each item containing a number of assets
covering a relatively small chunk of the nation. These assets are
“object\[s\] that \[contain\] a URI to data associated with the Item
that can be downloaded or streamed”, to quote [the
spec](https://github.com/radiantearth/stac-spec/blob/master/item-spec/item-spec.md#asset-object);
here, assets are things like “primary land cover classification” or “a
metadata object”.

To keep things simple, we’ll start off downloading data for a relatively
small region, namely North Carolina’s Ashe County. We’ll use data
included in the sf package to get the county’s geometry:

In [12]:
ashe <- sf::read_sf(system.file("shape/nc.shp", package = "sf"))[1, ]

sf::st_geometry(ashe) |> plot()

ERROR: Error in process_cpl_read_ogr(x, quiet, check_ring_dir = check_ring_dir, : package tibble not available: install first?


To filter our query down to just tiles intersecting this region, we’ll
need to provide the bounding box of the county in
[WGS84](https://en.wikipedia.org/wiki/World_Geodetic_System) as a query
parameter. We can use `sf::st_transform()` to reproject the county and
`sf::st_bbox()` to find our bounding box:

In [13]:
ashe_bbox <- ashe |>
  sf::st_transform(4326) |>
  sf::st_bbox()

ashe_bbox

ERROR: Error: object 'ashe' not found
